# Portfolio Optimisation: MPT, Efficient Frontier & Walk-Forward Backtesting (Corrected Exact Costs)

**Research question:** Does a long-only, capped maximum-Sharpe portfolio beat an equal-weight allocation and a minimum-variance strategy **out of sample**, after transaction costs?

**Methodology:** Part A uses **log returns** to estimate the MPT optimum. **Parts B–D** use daily **simple returns** for properly compounded, self-financing out-of-sample backtesting.

**Educational research only**, not investment advice. Five ETFs overlap (SPY/VTI/QQQ), historical mean returns are unstable, and a 2% fixed annual risk-free rate is illustrative. The 40% cap is a modelling assumption, not a recommendation.

## Setup and configuration
Run the optional installation cell once if required, then restart kernel and run all cells.

In [ ]:
%pip install -q numpy pandas scipy matplotlib yfinance

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize, linprog

TICKERS = ["SPY", "BND", "GLD", "QQQ", "VTI"]
YEARS = 5
DAYS_PER_YEAR = 252
RISK_FREE_RATE = 0.02  # illustrative annual simple rate
MAX_WEIGHT = 0.40
RANDOM_SEED = 42
N_RANDOM = 500
TRAIN_DAYS = 252 * 3        # rolling 3 trading years
REBALANCE_DAYS = 63         # every ~quarter
TRADING_COST_BPS = 10       # 0.10% of traded dollar notional
OUTPUT_DIR = Path('portfolio_results')
OUTPUT_DIR.mkdir(exist_ok=True)
print('Output directory:', OUTPUT_DIR.resolve())

## Part A — Adjusted prices, log returns, maximum Sharpe ratio

In [ ]:
import yfinance as yf

# Fixed historical dates make the analysis comparable to the CV backtest.
# Yahoo Finance downloads the adjusted prices directly; no local CSV is required.
start_date = "2021-10-18"
end_date = "2026-10-17"  # yfinance's end date is exclusive (last trading day: 2026-10-16)

adj_close_df = pd.DataFrame()
for ticker in TICKERS:
    data = yf.download(ticker, start=start_date, end=end_date,
                       auto_adjust=False, progress=False, threads=False)
    if data.empty:
        raise RuntimeError(f'No Yahoo Finance history returned for {ticker}')
    adjusted = data['Adj Close']
    if isinstance(adjusted, pd.DataFrame):
        adjusted = adjusted.iloc[:, 0]
    adj_close_df[ticker] = adjusted

adj_close_df = adj_close_df.sort_index().dropna(how='any')
if len(adj_close_df) <= TRAIN_DAYS + REBALANCE_DAYS:
    raise ValueError('Need more complete history to perform the requested walk-forward backtest.')
print('Aligned prices:', adj_close_df.shape, 'from', adj_close_df.index.min().date(),
      'to', adj_close_df.index.max().date())
display(adj_close_df.head())

In [ ]:
log_returns = np.log(adj_close_df / adj_close_df.shift(1)).dropna()
cov_matrix = log_returns.cov() * DAYS_PER_YEAR

def standard_deviation(weights, covariance):
    w = np.asarray(weights, dtype=float)
    return float(np.sqrt(max(w @ np.asarray(covariance) @ w, 0.0)))

def expected_return(weights, logarithmic_returns):
    return float(np.asarray(weights) @ (logarithmic_returns.mean().to_numpy() * DAYS_PER_YEAR))

def sharpe_ratio(weights, logarithmic_returns, covariance, risk_free_rate=RISK_FREE_RATE):
    vol = standard_deviation(weights, covariance)
    return (expected_return(weights, logarithmic_returns) - risk_free_rate) / vol if vol > 1e-12 else -np.inf

assert np.allclose(cov_matrix, cov_matrix.T)
display(cov_matrix)

### Shared optimiser
SLSQP is started from multiple feasible portfolios and feasibility is checked. This improves robustness but **does not constitute a proof of global optimality**.

In [ ]:
def validate_weights(w, cap=MAX_WEIGHT, atol=2e-5):
    w = np.asarray(w, dtype=float)
    return (np.all(np.isfinite(w)) and abs(w.sum()-1) <= atol
            and w.min() >= -atol and w.max() <= cap + atol)

def optimize_weights(mu, cov, objective='sharpe', cap=MAX_WEIGHT, rf=RISK_FREE_RATE):
    mu = np.asarray(mu, dtype=float)
    cov = np.asarray(cov, dtype=float)
    n = len(mu)
    if n * cap < 1 - 1e-10:
        raise ValueError('Infeasible weight cap for asset count')
    if not (np.all(np.isfinite(mu)) and np.all(np.isfinite(cov))):
        raise ValueError('Non-finite return/covariance estimates')
    cov = (cov + cov.T) / 2
    bounds = [(0., cap)] * n
    constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]
    if objective == 'sharpe':
        def loss(w):
            vol = np.sqrt(max(w @ cov @ w, 1e-18))
            return -(w @ mu - rf) / vol
    elif objective == 'min_variance':
        def loss(w):
            return w @ cov @ w
    else:
        raise ValueError('objective must be sharpe or min_variance')

    # Equal-weight and corner starts are always feasible whenever the cap permits it.
    starts = [np.full(n, 1/n)]
    for order in (np.argsort(-mu), np.argsort(mu)):
        trial = np.zeros(n)
        remaining = 1.
        for j in order:
            amount = min(cap, remaining)
            trial[j] = amount
            remaining -= amount
        starts.append(trial)
    best = None
    for x0 in starts:
        res = minimize(loss, x0=x0, method='SLSQP', bounds=bounds,
                       constraints=constraints, options={'maxiter': 1500, 'ftol': 1e-11})
        if res.success and validate_weights(res.x, cap):
            if best is None or loss(res.x) < loss(best):
                best = res.x
    if best is None:
        raise RuntimeError(f'{objective} optimisation did not converge to a feasible solution')
    return np.asarray(best)

# Annualised logarithmic-return parameters
mu_log = log_returns.mean().to_numpy() * DAYS_PER_YEAR
cov_log = cov_matrix.to_numpy()
optimal_weights = optimize_weights(mu_log, cov_log, 'sharpe')
optimal_portfolio_return = optimal_weights @ mu_log
optimal_portfolio_volatility = standard_deviation(optimal_weights, cov_log)
optimal_sharpe_ratio = (optimal_portfolio_return - RISK_FREE_RATE) / optimal_portfolio_volatility

print('In-sample maximum-Sharpe portfolio weights:')
display(pd.DataFrame({'ETF': TICKERS, 'Weight': optimal_weights}))
print(f'Expected annual log return: {optimal_portfolio_return:.2%}')
print(f'Annual log-return volatility: {optimal_portfolio_volatility:.2%}')
print(f'Model Sharpe ratio: {optimal_sharpe_ratio:.4f}')
assert validate_weights(optimal_weights)
assert np.isclose(optimal_sharpe_ratio, sharpe_ratio(optimal_weights, log_returns, cov_log))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(TICKERS, optimal_weights * 100)
ax.set_ylabel('Weight (%)')
ax.set_title('In-sample maximum-Sharpe allocation')
ax.set_ylim(0, MAX_WEIGHT * 100 + 5)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'optimal_weights.png', dpi=160)
plt.show()

## Part B — Efficient frontier and 500 random feasible portfolios
Returns and covariance here use the same logarithmic-return MPT estimates as Part A. The frontier is found by constrained minimum-variance optimisation at each target expected return; random portfolios are **only** comparison points.

In [ ]:
def random_feasible_weights(rng, n, cap=MAX_WEIGHT, max_tries=100000):
    if n * cap < 1 - 1e-12:
        raise ValueError('Infeasible cap')
    for _ in range(max_tries):
        w = rng.dirichlet(np.ones(n))
        if np.max(w) <= cap + 1e-12:
            return w
    raise RuntimeError('Could not sample feasible weights')

def efficient_frontier(mu, cov, cap=MAX_WEIGHT, points=60):
    mu = np.asarray(mu, float)
    cov = np.asarray(cov, float)
    n = len(mu)
    gmv = optimize_weights(mu, cov, objective='min_variance', cap=cap)
    # Linear programming accurately finds maximum feasible expected return.
    lp = linprog(-mu, A_eq=np.ones((1, n)), b_eq=[1.],
                 bounds=[(0, cap)]*n, method='highs')
    if not lp.success:
        raise RuntimeError('Maximum-return linear programme failed')
    upper_w = lp.x
    min_target, max_target = gmv @ mu, upper_w @ mu
    frontier_rows = []
    for t in np.linspace(min_target, max_target, points):
        if np.isclose(min_target, max_target):
            x0 = gmv.copy()
        else:
            fraction = np.clip((t-min_target)/(max_target-min_target), 0, 1)
            x0 = (1-fraction)*gmv + fraction*upper_w  # feasible target-return start
        cons = [{'type':'eq','fun': lambda w: w.sum()-1},
                {'type':'eq','fun': lambda w, target=t: w @ mu - target}]
        res = minimize(lambda w: w @ cov @ w, x0=x0,
                       method='SLSQP', bounds=[(0,cap)]*n,
                       constraints=cons, options={'maxiter': 2000, 'ftol': 1e-11})
        if res.success and validate_weights(res.x, cap) and abs(res.x @ mu - t) < 1e-5:
            frontier_rows.append({'Volatility': standard_deviation(res.x, cov), 'Return': res.x @ mu})
    return pd.DataFrame(frontier_rows), gmv

rng = np.random.default_rng(RANDOM_SEED)
random_w = np.array([random_feasible_weights(rng, len(TICKERS)) for _ in range(N_RANDOM)])
random_ret = random_w @ mu_log
random_vol = np.sqrt(np.einsum('bi,ij,bj->b', random_w, cov_log, random_w))
random_sharpe = (random_ret - RISK_FREE_RATE) / random_vol
frontier, gmv_weights = efficient_frontier(mu_log, cov_log)
assert len(frontier) > 0

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(random_vol, random_ret, c=random_sharpe, alpha=.65, s=18)
fig.colorbar(sc, ax=ax, label='Model Sharpe ratio')
ax.plot(frontier['Volatility'], frontier['Return'], linewidth=2.1, label='Upper efficient frontier')
ax.scatter(optimal_portfolio_volatility, optimal_portfolio_return, marker='*', s=260, label='Max Sharpe')
ax.scatter(standard_deviation(gmv_weights,cov_log), gmv_weights @ mu_log,
           marker='D', s=75, label='Minimum variance')
ax.set(xlabel='Annualised log-return volatility', ylabel='Expected annual log return',
       title='Efficient frontier and 500 feasible portfolios')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'efficient_frontier.png', dpi=160)
plt.show()

## Part C — Walk-forward, out-of-sample backtesting
**No look-ahead:** Each rebalance fits parameters on the preceding `TRAIN_DAYS` returns only. Holdings then drift during the following quarter. **Exact proportional transaction costs:** At a rebalance we solve for the post-trade NAV such that purchase/sale notional and fees are jointly self-financing. The first test-day return includes the initial cost, with no artificial cash financing. SPY is an uncapped *benchmark*, not a candidate for the 40%-cap optimisation. The model uses adjusted prices, assumes fractional shares and zero market impact, and treats 10 bps as an illustrative cost per dollar bought or sold.

In [ ]:
from scipy.optimize import brentq

simple_returns = adj_close_df.pct_change(fill_method=None).dropna()
assert (simple_returns > -1).all().all(), 'Bad adjusted-price return below -100%'

def self_financing_rebalance(pretrade_holdings, target_weights, cost_bps, pretrade_nav=None):
    """Find post-fee holdings H'=target*(NAV-fee), fee=k*sum(abs(H'-H)).

    The monotone cash balance equation has one root in [0, NAV].
    All turnover/cost figures reflect *actual dollar trades*, not approximate weights.
    """
    h = np.asarray(pretrade_holdings, dtype=float)
    target = np.asarray(target_weights, dtype=float)
    nav = float(h.sum()) if pretrade_nav is None else float(pretrade_nav)
    k = cost_bps / 10000.0
    if nav <= 0 or (h < 0).any() or h.sum() > nav + 1e-10 or not np.isclose(target.sum(), 1):
        raise ValueError('Invalid pretrade holdings or target')
    if not (0 <= k < 1):
        raise ValueError('Transaction cost must be in [0, 10000) bps')
    if k == 0:
        post_nav = nav
    else:
        def balance(post_nav):
            return post_nav + k * np.abs(target * post_nav - h).sum() - nav
        post_nav = brentq(balance, 0., nav, xtol=1e-14, rtol=1e-14)
    post = target * post_nav
    dollars_traded = float(np.abs(post - h).sum())
    fee = k * dollars_traded
    assert np.isclose(post.sum() + fee, nav, atol=1e-11), 'Trades must self-finance'
    assert np.isclose(fee, nav-post.sum(), atol=1e-11)
    return post, fee, dollars_traded / nav

def backtest_walk_forward(returns, train_days=TRAIN_DAYS, rebalance_days=REBALANCE_DAYS,
                          cap=MAX_WEIGHT, risk_free_rate=RISK_FREE_RATE,
                          cost_bps=TRADING_COST_BPS):
    r = returns.to_numpy(dtype=float)
    dates = returns.index
    n_days, n_assets = r.shape
    if n_assets * cap < 1 - 1e-10 or train_days < 100 or n_days <= train_days:
        raise ValueError('Invalid training window, cap, or insufficient data')
    if np.any(~np.isfinite(r)) or np.any(r <= -1):
        raise ValueError('Returns must be finite and greater than -100%')
    if not (0 <= cost_bps < 10000):
        raise ValueError('Transaction cost must be between 0 and 10000 bps')

    strategies = ['Max Sharpe', 'Minimum variance', 'Equal weight', 'SPY buy & hold']
    # Dollar-valued asset holdings; initialise with $1 cash and zero ETF holdings.
    equity = {name: 1.0 for name in strategies}
    holdings = {name: np.zeros(n_assets, dtype=float) for name in strategies}
    daily_growth = {name: [] for name in strategies}
    transactions = []
    decisions = []
    spy_idx = list(returns.columns).index('SPY')
    for block_start in range(train_days, n_days, rebalance_days):
        training = r[block_start-train_days:block_start]   # EXCLUDES test day
        mu = training.mean(axis=0) * DAYS_PER_YEAR
        cov = np.cov(training, rowvar=False) * DAYS_PER_YEAR
        target = {
            'Max Sharpe': optimize_weights(mu, cov, 'sharpe', cap=cap, rf=risk_free_rate),
            'Minimum variance': optimize_weights(mu, cov, 'min_variance', cap=cap, rf=risk_free_rate),
            'Equal weight': np.full(n_assets, 1/n_assets),
            'SPY buy & hold': np.eye(n_assets)[spy_idx],
        }
        for name in strategies:
            if name == 'SPY buy & hold' and block_start != train_days:
                continue  # no SPY rebalance after initial purchase
            if not validate_weights(target[name], cap=1. if name=='SPY buy & hold' else cap):
                raise AssertionError('Bad target portfolio')
            nav = equity[name]
            before = holdings[name]
            post_holdings, trading_cost, turnover = self_financing_rebalance(
                before, target[name], cost_bps, pretrade_nav=nav)
            equity[name] = float(post_holdings.sum())
            holdings[name] = post_holdings
            transactions.append({'Date': dates[block_start], 'Strategy':name,
                                 'Turnover': turnover, 'Cost': trading_cost})
            if name != 'SPY buy & hold':
                decisions.append({'Date': dates[block_start], 'Strategy': name,
                                  **dict(zip(returns.columns, target[name]))})
        for k in range(block_start, min(block_start+rebalance_days, n_days)):
            for name in strategies:
                prev_equity = equity[name]  # post-trade NAV for first day of block
                holdings[name] = holdings[name] * (1+r[k])
                equity[name] = float(holdings[name].sum())
                if equity[name] <= 0:
                    raise RuntimeError('Equity became nonpositive')
                # First day includes cost deducted *before* earning today's return.
                # Using previous day's pre-trade NAV for first day includes costs.
                daily_growth[name].append((dates[k], equity[name]))
        # Holdings always sum to NAV at block end.

    # Equity records include the exact fees on each rebalance date.
    equity_df = pd.DataFrame({name: pd.Series(dict(daily_growth[name])) for name in strategies})
    initial = pd.Series({name: 1.0 for name in strategies})
    # More precisely, first day return uses $1 as base; later days use prior EOD NAV.
    daily_r = equity_df.div(equity_df.shift(1)).sub(1)
    daily_r.iloc[0] = equity_df.iloc[0].div(initial).sub(1)
    return equity_df, daily_r, pd.DataFrame(transactions), pd.DataFrame(decisions)

equity_curve, strategy_returns, trading_log, allocation_log = backtest_walk_forward(simple_returns)
assert not strategy_returns.isna().any().any()
assert np.allclose((1+strategy_returns).cumprod().to_numpy(), equity_curve.to_numpy(), atol=1e-10)
print('First out-of-sample test date:', strategy_returns.index.min().date())
print('Last out-of-sample test date:', strategy_returns.index.max().date())
display(allocation_log.head())

### Performance and risk statistics
Annual return is geometric CAGR based on 252 trading days/year. Annualised volatility is the sample standard deviation of daily simple returns times √252. Sharpe uses the arithmetic daily excess return, with `RISK_FREE_RATE / 252` as the simple daily approximation. VaR/CVaR are *historical one-day loss* statistics at 95% confidence, not guarantees.

In [ ]:
def portfolio_statistics(returns, risk_free_rate=RISK_FREE_RATE):
    x = np.asarray(returns, dtype=float)
    if len(x) < 2 or not np.isfinite(x).all() or (x <= -1).any():
        raise ValueError('Invalid realised return observations')
    growth = np.cumprod(1 + x)
    years = len(x) / DAYS_PER_YEAR
    cagr = growth[-1] ** (1/years) - 1
    annual_vol = np.std(x, ddof=1) * np.sqrt(DAYS_PER_YEAR)
    sharpe = ((np.mean(x) - risk_free_rate / DAYS_PER_YEAR) * DAYS_PER_YEAR /
              annual_vol) if annual_vol > 1e-12 else np.nan
    previous_peak = np.maximum.accumulate(np.r_[1., growth])
    drawdown = np.r_[1., growth] / previous_peak - 1
    losses = -x
    value_at_risk = np.quantile(losses, .95)
    conditional_var = losses[losses >= value_at_risk].mean()
    downside = np.sqrt(np.mean(np.minimum(x - risk_free_rate/DAYS_PER_YEAR, 0.)**2))*np.sqrt(DAYS_PER_YEAR)
    return {'CAGR': cagr, 'Annualised volatility': annual_vol,
            'Sharpe': sharpe, 'Max drawdown': drawdown.min(),
            'Daily VaR 95% (loss)': value_at_risk,
            'Daily CVaR 95% (loss)': conditional_var,
            'Annual downside deviation': downside,
            'Terminal wealth / $1': growth[-1]}

performance = pd.DataFrame({name: portfolio_statistics(strategy_returns[name])
                            for name in strategy_returns.columns}).T
summary_turnover = trading_log.groupby('Strategy')['Turnover'].sum()
performance['Total turnover (sum of absolute trades)'] = summary_turnover
print('These are OUT-OF-SAMPLE realised results after hypothetical transaction costs:')
display(performance.round(4))
performance.to_csv(OUTPUT_DIR/'out_of_sample_performance.csv')
allocation_log.to_csv(OUTPUT_DIR/'allocation_history.csv', index=False)
trading_log.to_csv(OUTPUT_DIR/'transaction_log.csv', index=False)
# Exports enable independent recomputation of daily NAV, returns and prices.
adj_close_df.to_csv(OUTPUT_DIR/'adjusted_prices.csv')
log_returns.to_csv(OUTPUT_DIR/'daily_log_returns.csv')
strategy_returns.to_csv(OUTPUT_DIR/'out_of_sample_daily_returns.csv')
equity_curve.to_csv(OUTPUT_DIR/'out_of_sample_daily_nav.csv')


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 9), sharex=True)
equity_curve.plot(ax=axes[0], linewidth=1.7)
axes[0].set(ylabel='Portfolio value ($1 initial)', title='Out-of-sample growth of $1 — net of rebalancing costs')
axes[0].grid(alpha=.2)
for name in equity_curve.columns:
    wealth = equity_curve[name]
    drawdown = wealth / wealth.cummax().clip(lower=1.0) - 1.0
    drawdown.plot(ax=axes[1], label=name)
axes[1].set(ylabel='Drawdown', xlabel='Date', title='Out-of-sample drawdown')
axes[1].grid(alpha=.2)
axes[1].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR/'out_of_sample_performance.png', dpi=160)
plt.show()

## Part D — Stress periods and sensitivity to transaction costs
Stress: identify the **worst realised 63-trading-day period for the SPY benchmark** in the available out-of-sample period; evaluate the other strategies on the **same dates**, rather than selecting different stress periods for each portfolio. Cost sensitivity reruns the entire strategy at 0, 10, and 25 bps per dollar traded. These rates are illustrative, not market-validated estimates.

In [ ]:
stress_length = min(63, len(strategy_returns))
spy_rolling = (1+strategy_returns['SPY buy & hold']).rolling(stress_length).apply(np.prod, raw=True)-1
worst_end = spy_rolling.idxmin()
end_loc = strategy_returns.index.get_loc(worst_end)
start_loc = end_loc-stress_length+1
stress = strategy_returns.iloc[start_loc:end_loc+1]
stress_returns = (1+stress).prod()-1
print('SPY worst observed',stress_length,'trading days:',stress.index[0].date(), 'to', worst_end.date())
display(stress_returns.rename('Same-window cumulative return').to_frame())

cost_scenarios = {}
for bps in [0, 10, 25]:
    _, scenario_returns, _, _ = backtest_walk_forward(simple_returns, cost_bps=bps)
    cost_scenarios[f'{bps} bps'] = {
        name: portfolio_statistics(scenario_returns[name])['CAGR']
        for name in scenario_returns.columns
    }
cost_table = pd.DataFrame(cost_scenarios)
print('Out-of-sample CAGR by assumed transaction cost (per traded notional):')
display(cost_table.round(4))
cost_table.to_csv(OUTPUT_DIR/'transaction_cost_sensitivity.csv')

## Part E — Numerical verification (offline and live-data checks)
These assertions test the actual results and small synthetic cases. A successful SLSQP status and sampled-start checks do not prove global optimality; historical results do not predict future performance.

In [ ]:
# Live-data checks
assert len(strategy_returns) > 100
assert len(strategy_returns) == len(equity_curve)
assert not strategy_returns.isna().any().any()
assert np.isfinite(strategy_returns.to_numpy()).all()
assert (equity_curve > 0).all().all()
assert all(validate_weights(v) for v in allocation_log[TICKERS].to_numpy())
assert trading_log['Turnover'].ge(0).all()
assert trading_log['Cost'].ge(0).all()
assert np.allclose((1+strategy_returns).cumprod(), equity_curve, atol=1e-10)
assert np.allclose(equity_curve.pct_change().iloc[1:], strategy_returns.iloc[1:], atol=1e-12)

# Hand-verifiable exact fee: fully investing $1 cash at 10 bps yields
# post-fee NAV 1/(1+0.001), not 1-0.001 (the old approximation).
post, fee, turnover = self_financing_rebalance([0., 0.], [.5, .5], 10, pretrade_nav=1.)
assert np.isclose(post.sum(), 1/1.001, atol=1e-12)
assert np.isclose(fee, .001/1.001, atol=1e-12)
assert np.isclose(turnover, 1/1.001, atol=1e-12)
# Trading identical target holdings without price drift must generate no fee.
post2, fee2, turn2 = self_financing_rebalance(post, [.5, .5], 10)
assert np.allclose(post2, post) and abs(fee2) < 1e-12
# Exact fee identity with a non-trivial rebalance.
test_h = np.array([.8, .2])
post3, fee3, turn3 = self_financing_rebalance(test_h, [.2, .8], 25)
assert np.isclose(post3.sum() + fee3, 1.)
assert np.isclose(fee3, .0025 * np.abs(post3-test_h).sum())
# Exact hand-calculated $1, two-day buy-and-hold return; drift is handled by holdings values.
test_r = pd.DataFrame({'SPY':[0.10, -0.05], 'BND':[0., 0.]})
weights = np.array([.5,.5]); h = weights.copy()
for day in test_r.to_numpy(): h *= (1 + day)
assert np.isclose(h.sum(), 0.5*1.10*0.95 + 0.5)
assert validate_weights(np.array([.4,.4,.2]), cap=.4)
assert np.isclose(portfolio_statistics(np.zeros(252))['CAGR'], 0)
# Ensure all generated portfolios obey the cap and sum-to-one constraint.
assert all(validate_weights(w) for w in random_w)
assert np.isfinite(performance.select_dtypes('number').to_numpy()).all()
print('All portfolio feasibility, accounting, and numerical sanity checks passed.')
print('Saved result files in:', OUTPUT_DIR.resolve())

## Interpretation / GitHub discussion prompts

1. **In-sample versus out-of-sample:** Did maximum Sharpe actually beat equal weight and minimum variance after costs? Do not claim success unless observed.
2. **Estimation risk:** Five years total with three years training leaves roughly two years for out-of-sample testing. Evaluate how this affects confidence. SPY, VTI and QQQ are strongly overlapping exposures.
3. **Sensitivity:** Do changes in transaction costs or training periods materially change rankings?
4. **Limitations:** historical adjusted ETF prices; possible survivorship/selection bias from picking the five funds ex post; fixed illustrative risk-free rate; no slippage beyond assumed costs, tax, cash flows or borrow; daily historical data and trading at the rebalance boundary is an idealisation.
5. **For a CV:** Report *measured* out-of-sample differences, not the in-sample Sharpe estimate as realised performance.

**Saved**: `optimal_weights.png`, `efficient_frontier.png`, `out_of_sample_performance.png`, `out_of_sample_performance.csv`, `allocation_history.csv`, `transaction_log.csv`, and `transaction_cost_sensitivity.csv` in the folder printed above.

**Audit exports:** `adjusted_prices.csv`, `daily_log_returns.csv`, `out_of_sample_daily_nav.csv`, and `out_of_sample_daily_returns.csv` provide the data necessary to independently reproduce the calculations. The exact-cost convention treats fees as proportional to absolute dollar notional traded and assumes trades at adjusted close with fractional shares; it does not model spreads/impact or the intraday availability of close prices. Results use the fixed historical adjusted-price snapshot supplied in `portfolio_results/adjusted_prices.csv`.
